# LIBERO 在 LeRobot 中的接入方式

本 Notebook 演示并验证 LeRobot v0.5 对 LIBERO 仿真环境的集成：

1. 环境创建（LiberoEnvConfig + make_env）
2. Observation 结构验证（key 映射、shape）
3. Action Space 验证
4. Step 控制机制（PD 跟踪，非瞬移）
5. 轨迹跟踪演示
6. 观测图像可视化
7. obs 结构图生成

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. 环境创建

调用链：
```
make_env(LiberoEnvConfig)
  └→ create_libero_envs()           # lerobot/envs/libero.py
       └→ LiberoEnv(gym.Env)        # 封装 robosuite 的 OffScreenRenderEnv
            └→ OffScreenRenderEnv   # robosuite → MuJoCo
```

In [ ]:
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs.factory import make_env

env_cfg = LiberoEnvConfig(
    task="libero_spatial",
    task_ids=[0],              # 只取第一个任务，加速演示
    obs_type="pixels_agent_pos",
    observation_height=256,
    observation_width=256,
)

envs = make_env(env_cfg, n_envs=1)
env  = envs['libero_spatial'][0]

## 2. Observation 结构验证

LeRobot 将 robosuite 的原始 obs key 重新映射为嵌套 dict：

| LeRobot key | robosuite 原始 key | shape |
|---|---|---|
| `pixels/image` | `agentview_image` | (H, W, 3) |
| `pixels/image2` | `robot0_eye_in_hand_image` | (H, W, 3) |
| `robot_state/eef/pos` | `robot0_eef_pos` | (3,) |
| `robot_state/eef/quat` | `robot0_eef_quat` | (4,) |
| `robot_state/eef/mat` | 控制器旋转矩阵 | (3, 3) |
| `robot_state/gripper/qpos` | `robot0_gripper_qpos` | (2,) |
| `robot_state/joints/pos` | `robot0_joint_pos` | (7,) |

In [ ]:
obs, info = env.reset()

In [ ]:
obs, info = env.reset()

def print_obs_tree(d, prefix=""):
    """递归打印 obs 的 key 树和 shape"""
    for k, v in d.items():
        if isinstance(v, dict):
            print(f"{prefix}{k}/")
            print_obs_tree(v, prefix + "  ")
        elif isinstance(v, np.ndarray):
            print(f"{prefix}{k}: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"{prefix}{k}: type={type(v).__name__}")

print_obs_tree(obs)

In [ ]:
obs, _ = env.reset()

# MuJoCo/OpenGL 输出的图像上下和左右都是反的
# LeRobot 的 render() 内部做了 [::-1, ::-1]，但 obs 里的原始图像没有
img1 = obs["pixels"]["image"][0, ::-1, ::-1]
img2 = obs["pixels"]["image2"][0, ::-1, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img1)
axes[0].set_title(f"agentview (pixels/image)\nshape={obs['pixels']['image'].shape}")
axes[0].axis("off")

axes[1].imshow(img2)
axes[1].set_title(f"eye_in_hand (pixels/image2)\nshape={obs['pixels']['image2'].shape}")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/obs_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Action Space 验证

| 维度 | 含义 | 缩放后范围 |
|------|------|----------|
| `a[0:3]` | 末端位置 delta (x, y, z) | ±0.05 m/step |
| `a[3:6]` | 末端旋转 delta (axis-angle) | ±0.5 rad/step |
| `a[6]` | 夹爪控制 | -1=闭合, 1=打开 |

In [ ]:
action_space = env.single_action_space
print(f"Action space: {action_space}")
print(f"  shape: {action_space.shape}")
print(f"  low:   {action_space.low}")
print(f"  high:  {action_space.high}")

assert action_space.shape == (7,)
assert np.allclose(action_space.low, -1.0)
assert np.allclose(action_space.high, 1.0)
print("\n✓ Action space 验证通过: [-1, 1]^7")

## 4. Step 控制机制验证（PD 跟踪 vs 瞬移）

LIBERO 的 `step()` **不是瞬移**，而是基于 PD 控制的跟踪过程：

```
action → 缩放 → 目标增量 → OSC(PD控制器) → 力矩 → MuJoCo积分 → 实际位移
```

因此 **一步的实际位移 < 命令位移**（跟随比 α < 1）。

In [ ]:
obs, _ = env.reset()
eef_before = obs["robot_state"]["eef"]["pos"][0].copy()
print(f"初始 EEF 位置: {eef_before}")

# 发一个较大的 action（x 方向 +1）
action = np.zeros((1, 7), dtype=np.float32)
action[0, 0] = 1.0  # x 方向最大正向 → 缩放后 Δx = 0.05m

obs2, reward, terminated, truncated, info = env.step(action)
eef_after = obs2["robot_state"]["eef"]["pos"][0].copy()
print(f"Step 后 EEF 位置: {eef_after}")

delta_actual = eef_after - eef_before
delta_cmd = 0.05  # action=1.0 → 缩放后 0.05m
ratio = delta_actual[0] / delta_cmd

print(f"\n命令位移 Δx: {delta_cmd:.4f} m")
print(f"实际位移 Δx: {delta_actual[0]:.4f} m")


## 5. 轨迹跟踪演示

三个阶段：
1. 连续向 +x 方向运动 30 步
2. 停止（action=0）20 步
3. 向 -x 方向运动 30 步

观察 PD 控制的跟随特性和停止后的余振行为。

In [ ]:
# 需要用单个 env（非 vec_env）来做连续轨迹
env = env.envs[0]
obs, info = env.reset()
dummy_action = np.array([0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
for _ in range(20):
    obs, _, _, _, _ = env.step(dummy_action)

positions = [obs["robot_state"]["eef"]["pos"].copy()]
actions_record = []

# Phase 1: +x 30步
for _ in range(30):
    a = np.array([1.0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
    obs, _, _, _, _ = env.step(a)
    positions.append(obs["robot_state"]["eef"]["pos"].copy())
    actions_record.append(a[:3].copy())

# Phase 2: stop 20步
for _ in range(20):
    a = np.array([0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
    obs, _, _, _, _ = env.step(a)
    positions.append(obs["robot_state"]["eef"]["pos"].copy())
    actions_record.append(a[:3].copy())

# Phase 3: -x 30步
for _ in range(30):
    a = np.array([-1.0, 0, 0, 0, 0, 0, -1], dtype=np.float32)
    obs, _, _, _, _ = env.step(a)
    positions.append(obs["robot_state"]["eef"]["pos"].copy())
    actions_record.append(a[:3].copy())

positions = np.array(positions)
print(f"Total steps: {len(positions) - 1}")
print(f"Start pos: {positions[0]}")
print(f"End pos:   {positions[-1]}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

steps = np.arange(len(positions))
axes[0].plot(steps, positions[:, 0], "b-", linewidth=2, label="EEF x position")
axes[0].axvline(x=30, color="r", linestyle="--", alpha=0.5, label="Stop")
axes[0].axvline(x=50, color="g", linestyle="--", alpha=0.5, label="Reverse")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("X position (m)")
axes[0].set_title("EEF X Position Trajectory")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

actual_deltas = np.diff(positions[:, 0])
cmd_deltas = np.array([a[0] * 0.05 for a in actions_record])

axes[1].plot(range(len(actual_deltas)), actual_deltas, "b-", linewidth=1.5, label="Actual Δx")
axes[1].plot(range(len(cmd_deltas)), cmd_deltas, "r--", linewidth=1.5, label="Command Δx")
axes[1].axhline(y=0, color="k", linestyle="-", alpha=0.3)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Δx (m)")
axes[1].set_title("Actual vs Command Delta per Step")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

moving_alphas = actual_deltas[:30] / (cmd_deltas[:30] + 1e-10)
print(f"Moving phase mean α = {np.mean(moving_alphas):.4f}")

In [ ]:
env.close()
print("环境已关闭")